# Generating the engineer data for TalentMatch

Real staffing/bench data is private company data, so there is no public dataset of engineers
rated on these six things. I generate a realistic synthetic bench instead. To make it behave
like real engineers I add several sensible correlations rather than picking every number at
random.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker()
Faker.seed(42)
np.random.seed(42)

N = 2000  # how many engineers to make

## 2. Seniority, experience and domain

Seniority is the core trait. More senior people have more years of experience, and people with
more years tend to know their domain more deeply.

In [2]:
# seniority is roughly a bell curve
seniority = np.clip(np.random.normal(50, 20, N), 0, 99)

# more senior people have more years of experience
years = np.clip(seniority / 99 * 22 + np.random.normal(0, 2, N), 0, 30)

# more years usually means deeper domain knowledge
domain = np.clip(30 + years * 1.2 + np.random.normal(0, 14, N), 0, 99)

## 3. Client communication

Senior people tend to spend more time in front of clients.

In [3]:
communication = np.clip(0.5 * seniority + np.random.normal(25, 14, N), 0, 99)

## 4. Region and timezone overlap

Timezone overlap is measured against a US head office, so it depends on where the engineer is.

In [4]:
regions = np.random.choice(["Americas", "EMEA", "APAC"], N)

# how much each region overlaps with US working hours
region_overlap = {"Americas": 82, "EMEA": 55, "APAC": 28}
timezone_base = np.array([region_overlap[r] for r in regions])
timezone = np.clip(timezone_base + np.random.normal(0, 10, N), 0, 99)

## 5. Role and stack modernity

Only senior people are called architects. Some roles (ML, data, devops) work on more modern
stacks than others.

In [5]:
roles = np.random.choice(
    ["Backend Engineer", "Frontend Engineer", "Full-stack Engineer",
     "Data Engineer", "DevOps Engineer", "Solutions Architect", "ML Engineer"], N)

# an architect is a senior role, so don't call a junior person an architect
junior_architect = (roles == "Solutions Architect") & (seniority < 70)
roles[junior_architect] = "Backend Engineer"

# some roles skew towards newer technology
role_stack_bonus = {
    "ML Engineer": 22, "Data Engineer": 14, "DevOps Engineer": 12,
    "Solutions Architect": 5, "Backend Engineer": 0,
    "Frontend Engineer": 0, "Full-stack Engineer": 0,
}
stack_bonus = np.array([role_stack_bonus[r] for r in roles])
stack = np.clip(np.random.normal(48, 16, N) + stack_bonus, 0, 99)

## 6. Bandwidth

Very senior people are often spread across several projects, so they tend to have a bit less
free bandwidth.

In [6]:
bandwidth = np.clip(np.random.normal(65, 20, N) - seniority / 99 * 18, 0, 99)

## 7. Names and vertical

In [7]:
names = [fake.name() for _ in range(N)]
verticals = np.random.choice(["FinTech", "Healthcare", "E-commerce", "SaaS", "Gaming"], N)

## 8. Put it together and save

In [8]:
engineers = pd.DataFrame({
    "name": names,
    "role": roles,
    "region": regions,
    "vertical": verticals,
    "years_experience": np.round(years).astype(int),
    "seniority": np.round(seniority).astype(int),
    "domain": np.round(domain).astype(int),
    "communication": np.round(communication).astype(int),
    "timezone": np.round(timezone).astype(int),
    "stack": np.round(stack).astype(int),
    "bandwidth": np.round(bandwidth).astype(int),
})

engineers.to_csv("engineers.csv", index=False)
print("saved engineers.csv with", len(engineers), "engineers")
engineers.head()

saved engineers.csv with 2000 engineers


,name,role,region,vertical,years_experience,seniority,domain,communication,timezone,stack,bandwidth
0,Allison Hill,Backend Engineer,EMEA,SaaS,12,60,32,39,59,47,62
1,Noah Rhodes,DevOps Engineer,Americas,Gaming,10,47,42,40,94,73,49
2,Angie Henderson,Data Engineer,EMEA,SaaS,12,63,45,43,66,60,57
3,Daniel Wagner,ML Engineer,APAC,SaaS,17,80,57,58,7,55,53
4,Cristian Santos,Frontend Engineer,Americas,Gaming,6,45,18,45,70,67,52
